<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L51_Tool_Use_and_Function_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L51: Tool Use and Function Calling - APIs and Integration

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 51 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Define tool schemas (name, description, parameters) for an LLM
- Parse model output into tool calls and execute them
- Integrate results back into the conversation (function calling pattern)

---

## Concept: Function Calling

**Function calling**: model outputs structured tool calls (e.g. JSON with name and arguments); the app executes the function and returns the result; the result is appended to the conversation for the next turn. OpenAI and open models (e.g. Llama with tool layers) support this. We implement schema + parse + execute.

---

In [ ]:
!pip install transformers torch -q
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Tool Schema

---

In [ ]:
TOOL_SCHEMAS = [
    {"name": "get_weather", "description": "Get weather for a city", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}}},
    {"name": "calculate", "description": "Evaluate a math expression", "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}}},
]

def get_weather(city):
    return f"Weather in {city}: 72F, sunny."

def calculate(expression):
    try:
        return str(eval(expression))
    except:
        return "Error"

TOOLS = {"get_weather": get_weather, "calculate": calculate}
print("Tools:", list(TOOLS.keys()))

## Part 2: Parse and Execute

---

In [ ]:
def parse_tool_call(text):
    match = re.search(r"(\w+)\s*\(([^)]*)\)", text)
    if match:
        name, args_str = match.group(1), match.group(2)
        args = {"expression": args_str.strip()} if name == "calculate" else {"city": args_str.strip().strip('"')}
        return name, args
    return None, None

def execute_tool(name, args):
    if name in TOOLS:
        return TOOLS[name](**args)
    return "Unknown tool"

name, args = parse_tool_call("calculate(2+3)")
print("Parsed:", name, args)
print("Result:", execute_tool(name, args))

## Part 3: Chat Turn with Tool Result

---

In [ ]:
# Full flow: user message -> model may output tool_call -> execute -> append "Tool result: ..." -> model continues or answers.
print("Production: use chat API with tools=TOOL_SCHEMAS; handle response.choices[0].message.tool_calls.")

## Exercises

1. Add a third tool and its schema; extend parse_tool_call to handle it.
2. Format tool schemas as few-shot examples in the prompt for a model without native tool_calls.
3. Implement retry when the model outputs invalid JSON or unknown tool name.

---

## Key Takeaways

1. Tool schema = name, description, parameters (JSON Schema); helps model choose and format calls.
2. Parse model output → execute function → inject result into next message.
3. Native function calling (OpenAI, some open models) returns structured tool_calls; otherwise parse from text.

---

## Next Lesson

**L52: Production Deployment** – Serving, API, scaling.

---